<a href="https://colab.research.google.com/github/jacobiwoop/ComfyUi/blob/main/comfyui_by_a1k0_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

OPTIONS = {}
USE_GOOGLE_DRIVE = True
UPDATE_COMFY_UI = True
WORKSPACE = 'ComfyUI'
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if OPTIONS['USE_GOOGLE_DRIVE']:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

# Cloner ComfyUI si nécessaire
if not os.path.exists(WORKSPACE):
    print("-= Initial setup ComfyUI =-")
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}

# Aller dans le dossier
%cd {WORKSPACE}

# Mettre à jour si demandé
if OPTIONS['UPDATE_COMFY_UI']:
    print("-= Updating ComfyUI =-")
    !git pull

# Installer les dépendances
print("-= Install dependencies =-")
!pip install -r requirements.txt
!pip install xformers!=0.0.18


In [ ]:
# Crée les dossiers nécessaires
!mkdir -p ./models/checkpoints/
!mkdir -p ./models/vae/
!mkdir -p ./models/controlnet/
!mkdir -p ./models/upscale_models/
!mkdir -p ./models/loras/

# === SDXL (haute qualité + réalisme) ===
!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P ./models/checkpoints/
!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/resolve/main/sd_xl_refiner_1.0.safetensors -P ./models/checkpoints/

# === SD1.5 (léger + bon pour NSFW) ===
!wget -c https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors -P ./models/checkpoints/

# === VAE recommandé (meilleurs contrastes) ===
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

# === ControlNet utiles pour réalisme et poses ===
!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors -P ./models/controlnet/
!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11f1p_sd15_depth_fp16.safetensors -P ./models/controlnet/
!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_canny_fp16.safetensors -P ./models/controlnet/
!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_inpaint_fp16.safetensors -P ./models/controlnet/

# === Upscale (améliorer netteté, 2x ou 4x) ===
!wget -c https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth -P ./models/upscale_models/

# === Optionnel : Lora contrast/realism (décommente si besoin) ===
#!wget -c https://civitai.com/api/download/models/10638 -O ./models/loras/theovercomer8sContrastFix_sd15.safetensors


In [ ]:
# Téléchargement et installation de cloudflared (accès public à ComfyUI)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

# Fonction pour lancer cloudflared une fois que ComfyUI est prêt
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\n✅ ComfyUI est lancé, démarrage de cloudflared...\n")

    # Lance le tunnel cloudflared
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

    # Recherche l'URL publique dans la sortie
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com" in l:
            public_url = l[l.find("http"):]
            print("🌐 Accès public ComfyUI :", public_url, end='')
            break  # Arrête la boucle une fois l'URL trouvée

# Lance le thread pour surveiller et créer le tunnel
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# Lance ComfyUI
!python /content/drive/MyDrive/ComfyUI/main.py --dont-print-server



In [ ]:
# Désinstaller torchvision actuel
!pip uninstall -y torchvision

# Réinstaller une version compatible avec PyTorch 2.7.0
!pip install torchvision --index-url https://download.pytorch.org/whl/cu126


In [ ]:
!npm install -g localtunnel

import threading

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()




!python /content/drive/MyDrive/ComfyUI/main.py --dont-print-server


In [ ]:
!pip uninstall -y torchaudio
!pip install torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu126


Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu126
ERROR: Could not find a version that satisfies the requirement torchaudio==2.2.0 (from versions: 2.6.0+cu126, 2.7.0+cu126)
ERROR: No matching distribution found for torchaudio==2.2.0


In [ ]:
import subprocess
import threading
import time
import socket
import urllib.request  # ✅ Corrige l'erreur NameError

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\n✅ ComfyUI est prêt. Lancement de LocalTunnel...\n")

    # Affiche l'IP publique (utile pour whitelisting éventuel)
    public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("🌐 IP publique :", public_ip)

    # Lance LocalTunnel
    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
    time.sleep(1)  # Laisse LocalTunnel démarrer

    for line in p.stdout:
        l = line.decode()
        if 'your url is:' in l:
            print("🔗 URL publique ComfyUI :", l.split('your url is:')[-1].strip())
            break
        print(l, end='')

# Démarre le tunnel en arrière-plan
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# Lance ComfyUI
!python /content/drive/MyDrive/ComfyUI/main.py --dont-print-server

